# Notebook 2 — Model Training (nuScenes v1.0-trainval)

## Kaggle setup
1. Attach the dataset you saved from NB1 as input
2. Settings → Accelerator → GPU T4 ×2
3. Update `INPUT_DIR` in Cell 1 to match your NB1 output dataset path  
   e.g. `/kaggle/input/intentformer-nb1-out`
4. After training: Save Version → Output tab → New Dataset from Output

## Colab setup
1. Mount Drive — outputs from NB1 are already on Drive
2. `INPUT_DIR` and `OUTPUT_DIR` both point to `MyDrive/intentformer/`
3. No changes needed if you used the same Drive path in NB1

In [ ]:
# ── Cell 0: Detect environment + GPU check ───────────────────────────────────
import os, sys, torch

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB  = not IS_KAGGLE

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Colab'}")
print(f"PyTorch     : {torch.__version__}  |  CUDA: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Enable GPU T4 in Settings → Accelerator (Kaggle) "
                       "or Runtime → Change runtime type (Colab).")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# ── Cell 1: Imports + config ──────────────────────────────────────────────────
import os, sys, pickle, json, random, math, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm, trange

IS_KAGGLE = os.path.exists("/kaggle")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark     = True
# torch.backends.cudnn.deterministic = True  # disabled: conflicts with benchmark

# ── Paths ─────────────────────────────────────────────────────────────────────
if IS_KAGGLE:
    INPUT_DIR  = "/kaggle/input/datasets/asmitchatterjee/intentformer-nb1-out"   # ← UPDATE to your dataset slug
    OUTPUT_DIR = "/kaggle/working"
else:
    # Colab: mount Drive first if not already mounted
    if not os.path.exists("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    INPUT_DIR  = "/content/drive/MyDrive/intentformer"
    OUTPUT_DIR = "/content/drive/MyDrive/intentformer"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"INPUT_DIR  : {INPUT_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
assert os.path.exists(f"{INPUT_DIR}/trajectories.csv"), \
    f"trajectories.csv not found at {INPUT_DIR} — check INPUT_DIR path."

# ── Hyper-parameters ──────────────────────────────────────────────────────────
PAST_LEN    = 4;   FUTURE_LEN = 6;   DT = 0.5
D_MODEL     = 256; NHEAD = 4;        N_LAYERS = 3
K           = 5;   N_MAX_NEIGH = 5;  LIDAR_DIM = 7
BATCH_SIZE  = 512
LR          = 3e-4
EPOCHS      = 120    # 200 on mini; 120 fits T4 12hr budget on trainval
PATIENCE    = 20
GRAD_CLIP   = 1.0
DEVICE      = "cuda"

LAMBDA_INT        = 0.15
LAMBDA_DIV_MAX    = 0.10
DIV_WARMUP_EP     = 20
DIV_RAMP_EP       = 60
LAMBDA_INT_WARMUP = 10

def get_lambda_div(ep):
    if ep <= DIV_WARMUP_EP: return 0.0
    return min(LAMBDA_DIV_MAX, LAMBDA_DIV_MAX * (ep - DIV_WARMUP_EP) / DIV_RAMP_EP)

def get_lambda_int(ep):
    return 0.0 if ep <= LAMBDA_INT_WARMUP else LAMBDA_INT

print("Config OK")


In [ ]:
# ── Cell 2: Model architecture ────────────────────────────────────────────────

class TemporalEncoder(nn.Module):
    def __init__(self, input_dim=4, d_model=256, nhead=4, n_layers=3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.register_buffer("pe", self._build_pe(64, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    @staticmethod
    def _build_pe(max_len, d_model):
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)

    def forward(self, x):
        T   = x.size(1)
        emb = self.input_proj(x) + self.pe[:, :T, :]
        out = self.transformer(emb)
        return self.norm(out[:, -1, :])


class SocialAttention(nn.Module):
    def __init__(self, d_model=256, nhead=4):
        super().__init__()
        self.neighbour_enc = nn.Linear(4, d_model)
        self.cross_attn    = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.gate_fc       = nn.Linear(d_model, 1)
        self.out_norm      = nn.LayerNorm(d_model)

    def forward(self, focal_h, neighbours):
        B, N, T, _ = neighbours.shape
        neigh_mask  = (neighbours.abs().sum(dim=(-1, -2)) > 1e-6)
        if not neigh_mask.any(): return focal_h
        neigh_emb = self.neighbour_enc(neighbours.view(B * N, T, -1))
        focal_q   = focal_h.unsqueeze(1).expand(-1, N, -1).reshape(B * N, 1, -1)
        kpm       = torch.zeros(B * N, T, dtype=torch.bool, device=focal_h.device)
        ctx, _    = self.cross_attn(focal_q, neigh_emb, neigh_emb, key_padding_mask=kpm)
        ctx       = ctx.view(B, N, -1)
        gate_scores = self.gate_fc(ctx).squeeze(-1)
        dtype_orig  = gate_scores.dtype
        gate_scores = gate_scores.float().masked_fill(~neigh_mask, -1e9)
        gate_w      = torch.softmax(gate_scores, dim=1).to(dtype_orig).unsqueeze(-1)
        social      = (gate_w * ctx).sum(dim=1)
        has = neigh_mask.any(dim=1)
        return torch.where(has.unsqueeze(-1), self.out_norm(social + focal_h), focal_h)


class IntentHead(nn.Module):
    def __init__(self, d_model=256, n_intents=5):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_intents))
    def forward(self, h): return self.fc(h)


class GMMHead(nn.Module):
    def __init__(self, d_model=256, K=5, future_steps=6):
        super().__init__()
        self.K = K; self.T = future_steps
        self.mode_heads = nn.ModuleList(
            [nn.Linear(d_model, future_steps * 5) for _ in range(K)])
        self.pi_head = nn.Linear(d_model, K)
        with torch.no_grad():
            for head in self.mode_heads:
                for t in range(future_steps):
                    head.bias[5*t+2].fill_(2.0); head.bias[5*t+3].fill_(2.0)

    def forward(self, h):
        B = h.size(0); mu_l, sig_l, rho_l = [], [], []
        for head in self.mode_heads:
            o = head(h).view(B, self.T, 5)
            mu_l.append(o[..., :2])
            sig_l.append(F.softplus(o[..., 2:4]) + 1e-3)
            rho_l.append(torch.tanh(o[..., 4]) * 0.9)
        return (torch.stack(mu_l, dim=1), torch.stack(sig_l, dim=1),
                torch.stack(rho_l, dim=1), torch.softmax(self.pi_head(h), dim=-1))


class IntentFormer(nn.Module):
    def __init__(self, d_model=256, nhead=4, n_layers=3, K=5, future_steps=6, lidar_dim=7):
        super().__init__()
        self.temporal_enc = TemporalEncoder(4, d_model, nhead, n_layers)
        self.social_attn  = SocialAttention(d_model, nhead)
        self.lidar_proj   = nn.Sequential(nn.Linear(6, 64), nn.ReLU(), nn.Linear(64, d_model))
        self.fusion_norm  = nn.LayerNorm(d_model)
        self.intent_head  = IntentHead(d_model)
        self.gmm_head     = GMMHead(d_model, K, future_steps)

    def forward(self, past, neighbours, lidar_input):
        h = self.temporal_enc(past)
        h = self.social_attn(h, neighbours)
        h = self.fusion_norm(h + self.lidar_proj(lidar_input[:, :6]) * lidar_input[:, 6:7])
        mu, sigma, rho, pi = self.gmm_head(h)
        return mu, sigma, rho, pi, self.intent_head(h.detach())

print("✓ Architecture defined.")


In [ ]:
# ── Cell 3: Loss functions + metrics ──────────────────────────────────────────
def denorm(xy, std_x, std_y):
    out = xy.clone(); out[..., 0] *= std_x; out[..., 1] *= std_y; return out

@torch.amp.custom_fwd(cast_inputs=torch.float32, device_type="cuda")
def bivariate_nll(gt, mu, sigma, rho):
    eps  = 1e-6
    sx   = sigma[..., 0].clamp(min=eps); sy = sigma[..., 1].clamp(min=eps)
    dx   = (gt[..., 0] - mu[..., 0]) / sx
    dy   = (gt[..., 1] - mu[..., 1]) / sy
    r    = rho.clamp(-0.9, 0.9)
    z    = dx**2 + dy**2 - 2 * r * dx * dy
    omr2 = (1 - r**2).clamp(min=1e-4)
    log_p = (-0.5 * z / omr2 - torch.log(sx) - torch.log(sy)
              - 0.5 * torch.log(omr2) - math.log(2 * math.pi))
    return -log_p.mean(dim=-1)

@torch.amp.custom_fwd(cast_inputs=torch.float32, device_type="cuda")
def gmm_nll_loss(gt, mu, sigma, rho, pi):
    log_terms = [torch.log(pi[:, k].clamp(1e-6)) - bivariate_nll(gt, mu[:,k], sigma[:,k], rho[:,k])
                 for k in range(pi.shape[1])]
    return -torch.logsumexp(torch.stack(log_terms, dim=1), dim=1).mean()

def intent_ce_loss(logits, targets): return F.cross_entropy(logits, targets)

@torch.amp.custom_fwd(cast_inputs=torch.float32, device_type="cuda")
def diversity_loss(mu, std_x, std_y):
    mu_r  = denorm(mu.float(), std_x, std_y)
    K_    = mu_r.shape[1]; loss = torch.zeros(1, device=mu.device); count = 0
    for i in range(K_):
        for j in range(i + 1, K_):
            dist = torch.norm(mu_r[:, i] - mu_r[:, j], dim=-1).mean(dim=-1)
            loss = loss + torch.exp(-dist).mean(); count += 1
    return loss / max(count, 1)

def min_ade_fde(mu, gt, std_x, std_y):
    mu_r = denorm(mu, std_x, std_y); gt_r = denorm(gt, std_x, std_y)
    dist = torch.norm(mu_r - gt_r.unsqueeze(1).expand_as(mu_r), dim=-1)
    return dist.mean(-1).min(1).values.mean().item(), dist[:,-1].min(1).values.mean().item()

def pi_ade(mu, gt, pi, std_x, std_y):
    B = mu.size(0); k = pi.argmax(1); sel = mu[torch.arange(B), k]
    return torch.norm(denorm(sel, std_x, std_y) - denorm(gt, std_x, std_y), dim=-1).mean().item()

def cv_ade(past, mu, gt, std_x, std_y):
    B = mu.size(0); vel = past[:,-1,:2] - past[:,-2,:2]
    cv_end_r = denorm((past[:,-1,:2] + vel * FUTURE_LEN).unsqueeze(1), std_x, std_y).squeeze(1)
    mu_r = denorm(mu, std_x, std_y)
    k    = torch.norm(mu_r[:,:,-1,:] - cv_end_r.unsqueeze(1), dim=-1).argmin(1)
    return torch.norm(mu_r[torch.arange(B), k] - denorm(gt, std_x, std_y), dim=-1).mean().item()

def mode_collapse_rate(pi, thresh=0.9):
    return (pi.max(dim=1).values > thresh).float().mean().item()

def intent_accuracy(logits, targets):
    return (logits.argmax(1) == targets).float().mean().item()

print("✓ Loss functions defined.")


In [ ]:
# ── Cell 4: Dataset ───────────────────────────────────────────────────────────
class TrajectoryDataset(Dataset):
    def __init__(self, df, neighbors_map, lidar_map, split, augment=False):
        self.augment = augment
        allowed = set(df[df["split"] == split]["sample_id"].unique())
        grouped = {sid: grp.sort_values("frame_rel")
                   for sid, grp in df[df["sample_id"].isin(allowed)]
                                     .groupby("sample_id", sort=False)}
        self.items = []
        for sid, sdf in grouped.items():
            past_r = sdf[sdf["frame_rel"] < PAST_LEN][["x_norm","y_norm"]].values
            fut_r  = sdf[sdf["frame_rel"] >= PAST_LEN][["x_norm","y_norm"]].values
            if len(past_r) != PAST_LEN or len(fut_r) != FUTURE_LEN: continue
            diffs     = np.diff(past_r, axis=0)
            vel       = np.vstack([diffs[:1], diffs]).astype(np.float32) / DT
            past_feat = np.concatenate([past_r, vel], axis=-1).astype(np.float32)
            intent_idx = int(sdf["intent_idx"].iloc[0])
            neigh = neighbors_map.get(sid, np.zeros((N_MAX_NEIGH, PAST_LEN, 4), np.float32))
            entry = lidar_map.get(sid)
            if isinstance(entry, dict):
                feat, valid = entry["feat"], float(entry["valid"])
            else:
                feat = entry if entry is not None else np.zeros(6, np.float32); valid = 1.0
            lidar_7d = np.append(feat, valid).astype(np.float32)
            self.items.append((past_feat.copy(), fut_r.astype(np.float32).copy(),
                               neigh.copy(), lidar_7d, intent_idx))

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        past_feat, fut_r, neigh, lidar_7d, intent_idx = self.items[i]
        if self.augment:
            past_feat, fut_r, neigh = past_feat.copy(), fut_r.copy(), neigh.copy()
            if random.random() < 0.5:
                past_feat[:,0] *= -1; past_feat[:,2] *= -1
                fut_r[:,0] *= -1; neigh[:,:,0] *= -1; neigh[:,:,2] *= -1
            if random.random() < 0.5:
                s = np.float32(random.uniform(0.8, 1.2))
                past_feat[:,:2] *= s; past_feat[:,2:] *= s; fut_r *= s; neigh *= s
            neigh[np.random.rand(N_MAX_NEIGH) < 0.3] = 0.0
        return (torch.tensor(past_feat), torch.tensor(fut_r, dtype=torch.float32),
                torch.tensor(neigh,      dtype=torch.float32),
                torch.tensor(lidar_7d,   dtype=torch.float32), intent_idx)


In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────────────
df         = pd.read_csv(f"{INPUT_DIR}/trajectories.csv")
norm_stats = pd.read_csv(f"{INPUT_DIR}/norm_stats.csv", index_col=0).squeeze()
std_x      = float(norm_stats["std_x"]); std_y = float(norm_stats["std_y"])
print(f"std_x={std_x:.4f}  std_y={std_y:.4f}")

with open(f"{INPUT_DIR}/neighbors.pkl",   "rb") as fh: neighbors_map = pickle.load(fh)
with open(f"{INPUT_DIR}/lidar_feats.pkl", "rb") as fh: lidar_map     = pickle.load(fh)

train_ds = TrajectoryDataset(df, neighbors_map, lidar_map, "train", augment=True)
val_ds   = TrajectoryDataset(df, neighbors_map, lidar_map, "val",   augment=False)
test_ds  = TrajectoryDataset(df, neighbors_map, lidar_map, "test",  augment=False)

LOADER_KW = dict(num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  **LOADER_KW)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, **LOADER_KW)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, **LOADER_KW)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
assert len(val_ds) > 0 and len(test_ds) > 0


In [ ]:
# ── Cell 6: Model + optimiser + AMP ──────────────────────────────────────────
def unwrap(m):
    """Return the raw nn.Module, stripping DataParallel and torch.compile wrappers.
    Use this before torch.save so the checkpoint has clean keys (no 'module.' prefix).
    """
    if hasattr(m, 'module'):    m = m.module    # DataParallel
    if hasattr(m, '_orig_mod'): m = m._orig_mod  # torch.compile
    return m

model = IntentFormer(D_MODEL, NHEAD, N_LAYERS, K, FUTURE_LEN, LIDAR_DIM)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    # torch.compile + DataParallel is incompatible: DataParallel can't deep-copy
    # an OptimizedModule across devices → AttributeError on temporal_enc / any submodule.
    # Use DataParallel only; AMP + large batch size already saturates both T4s.
    print(f"Using {n_gpus} GPUs (DataParallel; torch.compile skipped for multi-GPU)")
    model = nn.DataParallel(model)
else:
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("✓ torch.compile active (single GPU)")
    except Exception as e:
        print(f"torch.compile skipped ({e})")

model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler    = torch.amp.GradScaler("cuda")

def cosine_warmup(opt, warmup=10, total=EPOCHS):
    def lr_fn(ep):
        if ep < warmup: return ep / warmup
        p = (ep - warmup) / (total - warmup)
        return 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

scheduler = cosine_warmup(optimizer)
print("✓ AMP GradScaler ready")


In [ ]:
# ── Cell 7: Training loop ─────────────────────────────────────────────────────
best_val_pi_ade = float("inf")
no_improve = 0; history = []; n_batches = len(train_loader)

for epoch in trange(1, EPOCHS + 1, desc="Epochs"):
    lam_div = get_lambda_div(epoch); lam_int = get_lambda_int(epoch)
    model.train()
    e_loss = e_gmm = e_div = e_int = 0.0
    t0 = time.perf_counter()

    for past, future, neigh, lidar, intent_tgt in train_loader:
        past       = past.to(DEVICE, non_blocking=True)
        future     = future.to(DEVICE, non_blocking=True)
        neigh      = neigh.to(DEVICE, non_blocking=True)
        lidar      = lidar.to(DEVICE, non_blocking=True)
        intent_tgt = intent_tgt.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):           # runs forward in float16
            mu, sigma, rho, pi, intent_logits = model(past, neigh, lidar)
            L_gmm = gmm_nll_loss(future, mu, sigma, rho, pi)
            L_div = diversity_loss(mu, std_x, std_y)
            L_int = intent_ce_loss(intent_logits, intent_tgt)
            loss  = L_gmm + lam_div * L_div + lam_int * L_int
        if not torch.isfinite(loss): continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        e_loss += loss.item()/n_batches; e_gmm += L_gmm.item()/n_batches
        e_div  += L_div.item()/n_batches; e_int += L_int.item()/n_batches

    ep_ips = n_batches / (time.perf_counter() - t0)
    model.eval()
    v_min_ade = v_min_fde = v_pi_ade = v_cv_ade = v_collapse = v_int_acc = 0.0
    with torch.no_grad():
        for past, future, neigh, lidar, intent_tgt in val_loader:
            past      = past.to(DEVICE, non_blocking=True)
            future     = future.to(DEVICE, non_blocking=True)
            neigh      = neigh.to(DEVICE, non_blocking=True)
            lidar      = lidar.to(DEVICE, non_blocking=True)
            intent_tgt = intent_tgt.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                mu, sigma, rho, pi, il = model(past, neigh, lidar)
            a, f = min_ade_fde(mu, future, std_x, std_y)
            v_min_ade  += a / len(val_loader)
            v_min_fde  += f / len(val_loader)
            v_pi_ade   += pi_ade(mu, future, pi, std_x, std_y)   / len(val_loader)
            v_cv_ade   += cv_ade(past, mu, future, std_x, std_y) / len(val_loader)
            v_collapse += mode_collapse_rate(pi)                   / len(val_loader)
            v_int_acc  += intent_accuracy(il, intent_tgt)          / len(val_loader)

    scheduler.step()
    history.append({"epoch": epoch, "loss": e_loss, "gmm": e_gmm, "div": e_div, "int": e_int,
                    "val_min_ade": v_min_ade, "val_min_fde": v_min_fde,
                    "val_pi_ade": v_pi_ade, "val_cv_ade": v_cv_ade,
                    "val_collapse": v_collapse, "val_int_acc": v_int_acc})

    if np.isfinite(v_pi_ade) and v_pi_ade < best_val_pi_ade:
        best_val_pi_ade = v_pi_ade; no_improve = 0
        torch.save(unwrap(model).state_dict(), f"{OUTPUT_DIR}/best_intentformer.pt")  # strip DataParallel prefix
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch}."); break

    if epoch % 10 == 0:
        print(f"Ep {epoch:3d} | gmm {e_gmm:.3f} | minADE {v_min_ade:.4f}m | "
              f"pi-ADE {v_pi_ade:.4f}m | collapse {v_collapse:.1%} | "
              f"it/s {ep_ips:.1f} | best {best_val_pi_ade:.4f}m")

pd.DataFrame(history).to_csv(f"{OUTPUT_DIR}/training_history.csv", index=False)
assert os.path.exists(f"{OUTPUT_DIR}/best_intentformer.pt"), "Checkpoint never saved."
print(f"\n✓ Done. Best val pi-ADE: {best_val_pi_ade:.4f} m")
if IS_KAGGLE:
    print("→ Save Version → Output tab → New Dataset from Output")


In [ ]:
# ── Cell 8: CV baseline + test eval ──────────────────────────────────────────
def constant_velocity_pred(past):
    pos = past[:,-1,:2].clone().float(); vel = (past[:,-1,:2] - past[:,-2,:2]).float()
    return torch.cat([(pos := pos + vel).unsqueeze(1) for _ in range(FUTURE_LEN)], dim=1)

# Reload into a fresh bare IntentFormer — the checkpoint has no 'module.' prefix
# (we saved unwrap(model).state_dict() above), so don't load into the DataParallel model.
eval_model = IntentFormer(D_MODEL, NHEAD, N_LAYERS, K, FUTURE_LEN, LIDAR_DIM).to(DEVICE)
eval_model.load_state_dict(torch.load(
    f"{OUTPUT_DIR}/best_intentformer.pt", map_location=DEVICE, weights_only=True))
eval_model.eval()
model = eval_model  # rebind name so the rest of the cell works unchanged

cv_ade_s = cv_fde_s = m_min_ade = m_min_fde = m_pi_ade_s = m_cv_ade_s = 0.0
n = len(test_loader)
with torch.no_grad():
    for past, future, neigh, lidar, _ in tqdm(test_loader, desc="Test eval"):
        past, future = past.to(DEVICE), future.to(DEVICE)
        neigh, lidar = neigh.to(DEVICE), lidar.to(DEVICE)
        cv_pred_r = denorm(constant_velocity_pred(past), std_x, std_y)
        fut_r     = denorm(future, std_x, std_y)
        cv_dist   = torch.norm(cv_pred_r - fut_r, dim=-1)
        cv_ade_s += cv_dist.mean(-1).mean().item() / n
        cv_fde_s += cv_dist[:,-1].mean().item()   / n
        with torch.amp.autocast("cuda"):
            mu, sigma, rho, pi, _ = model(past, neigh, lidar)
        a, f = min_ade_fde(mu, future, std_x, std_y)
        m_min_ade  += a / n; m_min_fde  += f / n
        m_pi_ade_s += pi_ade(mu, future, pi, std_x, std_y) / n
        m_cv_ade_s += cv_ade(past, mu, future, std_x, std_y) / n

print(f"CV baseline   ADE: {cv_ade_s:.4f} m | FDE: {cv_fde_s:.4f} m")
print(f"IntentFormer minADE@{K}: {m_min_ade:.4f} m | minFDE@{K}: {m_min_fde:.4f} m")
print(f"pi-ADE: {m_pi_ade_s:.4f} m | cv-ADE: {m_cv_ade_s:.4f} m")
print(f"Improvement vs CV: {(cv_ade_s - m_pi_ade_s)/cv_ade_s*100:.1f}%")

with open(f"{OUTPUT_DIR}/cv_baseline.json", "w") as fh:
    json.dump({"cv_ade_m": cv_ade_s, "cv_fde_m": cv_fde_s,
               "model_min_ade_m": m_min_ade, "model_min_fde_m": m_min_fde,
               "model_pi_ade_m": m_pi_ade_s, "model_cv_ade_m": m_cv_ade_s}, fh)


In [ ]:
# ── Cell 9: Training curves ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
hist = pd.read_csv(f"{OUTPUT_DIR}/training_history.csv")
fig, axes = plt.subplots(1, 4, figsize=(24, 4))
axes[0].plot(hist["epoch"], hist["gmm"], label="GMM-NLL")
axes[0].plot(hist["epoch"], hist["div"], label="Diversity", ls="--")
axes[0].plot(hist["epoch"], hist["int"], label="Intent CE", ls=":")
axes[0].set_title("Training losses"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist["epoch"], hist["val_min_ade"], label=f"minADE@{K} (oracle)")
axes[1].plot(hist["epoch"], hist["val_pi_ade"],  label="pi-ADE ← ckpt", ls="--")
axes[1].plot(hist["epoch"], hist["val_cv_ade"],  label="cv-ADE", ls=":")
axes[1].set_title("Val ADE (m)"); axes[1].legend(); axes[1].grid(True)
axes[2].plot(hist["epoch"], hist["val_collapse"]*100, color="red")
axes[2].axhline(20, color="gray", ls="--", label="20% target")
axes[2].set_title("Mode collapse %"); axes[2].legend(); axes[2].grid(True)
axes[3].plot(hist["epoch"], hist["val_int_acc"]*100, color="purple")
axes[3].axhline(20, color="gray", ls="--", label="random=20%")
axes[3].set_title("Intent accuracy %"); axes[3].legend(); axes[3].grid(True)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Saved training_curves.png")
